# شام — مسار جمع وتدريب أداة ترميز الصوت (VQ-VAE) — CPU، بلا حاجة لـ GPU

دفتر **مستقل تماماً** عن مساري النص (A وB) وعن دفتر المرحلة الثانية (الذي يحتاج GPU) — هدفه الوحيد بناء أداة ترميز صوت (audio tokenizer) حقيقية عالية الجودة، تدريجياً، عبر تشغيلات متكررة غير محدودة على CPU المجاني.

**لماذا مستقل؟** أداة ترميز الصوت (VQ-VAE) تتعلم من الصوت نفسه فقط (بلا إشراف، بلا حاجة للنموذج الرئيسي أو نقطة حفظه) — تدريبها لا يحتاج GPU إطلاقاً ولا يستهلك أي حصة GPU المحدودة (30 ساعة/أسبوع). هذا يجعله مساراً يعمل يومياً دون توقف، تماماً كمساري النص.

كل تشغيل: يجمع دفعة **جديدة** من تسجيلات Common Voice العربية الحقيقية (مرخّصة CC0) لم تُستخدم في التشغيلات السابقة، يكمل تدريب نفس أداة الترميز من نقطة توقفها، وينشرها كمجموعة بيانات Kaggle خاصة بهذا المسار — جاهزة ليستخدمها لاحقاً دفتر التدريب متعدد الوسائط (بدل تدريب أداة الترميز من الصفر داخل ذلك الدفتر، فيوفَّر وقت الـGPU النادر للتدريب الفعلي فقط).

## قبل "Save Version → Save & Run All":
1. **فعّل الإنترنت** من Settings (لا حاجة لـ GPU — اتركه Off لتوفير حصتك).
2. تأكد من وجود نفس أسرار Kaggle: `GITHUB_TOKEN`، `KAGGLE_USERNAME`، `KAGGLE_KEY` في Add-ons → Secrets.
3. **من التشغيل الثاني فصاعداً**: أضف نتاج هذا الدفتر نفسه (Notebook Output) كـ Input — بنفس الطريقة المستخدمة مع مساري النص — كي يكمل التدريب بدل البدء من الصفر كل مرة.
4. استخدم **Save Version → Save & Run All (Commit)** دائماً، ويمكن جدولته (Schedule) للتشغيل التلقائي اليومي.


### 1) سحب الكود الحقيقي من GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py"))
sys.path.insert(0, CODE_DIR)
print("كود شام الحقيقي جاهز في:", CODE_DIR)


### 2) تثبيت المكتبات الإضافية

In [ ]:
try:
    import datasets
    print(f"مكتبة datasets متوفرة مسبقاً (نسخة {datasets.__version__}).")
except ImportError:
    subprocess.run(["pip", "install", "-q", "datasets"], check=True)
    print("تم تثبيت مكتبة datasets.")

try:
    import soundfile
    print("مكتبة soundfile متوفرة مسبقاً.")
except ImportError:
    subprocess.run(["pip", "install", "-q", "soundfile"], check=True)
    print("تم تثبيت مكتبة soundfile.")


### 3) استئناف أداة ترميز الصوت من التشغيل السابق (إن وُجد)

يبحث عن `audio_tokenizer.pt` و`progress.json` (يحفظان معاً — انظر خلية النشر أدناه) في أي Input مرفق. إن لم يجد شيئاً، يبدأ أداة ترميز جديدة بأوزان عشوائية (متوقّع فقط في أول تشغيل حقيقي).

الحجم المستخدم هنا هو **الحجم الإنتاجي الكامل** (`n_mels=80`، عدد الرموز = 2048 — القيم الافتراضية في `audio_tokenizer.py` نفسها) وليس نسخة "بداية" مصغّرة، لأن هذا المسار مخصص بالذات لبناء أداة الترميز النهائية تدريجياً عبر تشغيلات متكررة غير محدودة (بخلاف دفتر المرحلة الثانية الذي استخدم حجماً مصغّراً كبداية آمنة لأنه محدود بحصة GPU).

In [ ]:
import json as _json
from pathlib import Path
from audio_tokenizer import AudioTokenizer, AudioTokenizerConfig
from tokenizer_select import select_pretrained_tokenizer, _progress_samples

# الاستئناف: يُفحص كل ما في المدخلات ويُختار الأكثر تدريباً؛ النسخ غير المتوافقة
# (قاموس بحجم مختلف، أو محفوظة بإصدار أقدم) تُصلَح وتُقبل ولا تُتجاهل (tokenizer_select.py).
samples_consumed = 0
start_step = 0
print("البحث عن نسخة سابقة من أداة الترميز (audio):")
_picked = select_pretrained_tokenizer("audio")
if _picked:
    audio_tokenizer, start_step, _resume_path = _picked
    audio_tokenizer_cfg = audio_tokenizer.cfg
    samples_consumed = _progress_samples(_resume_path, "audio")
    print(f"استؤنف من: {_resume_path} (خطوة {start_step:,}، {samples_consumed:,} عيّنة سابقاً)")
else:
    audio_tokenizer_cfg = AudioTokenizerConfig()
    audio_tokenizer = AudioTokenizer(audio_tokenizer_cfg)
    print("لا توجد نسخة سابقة صالحة — بدء أداة ترميز جديدة (متوقَّع فقط في أول تشغيل).")


### 4) جمع دفعة جديدة من تسجيلات Common Voice العربية الحقيقية

`skip=samples_consumed` يضمن أن كل تشغيل يرى عينات **جديدة** لم يرها التشغيل السابق (لا نكرر نفس أول 3000 عينة إلى الأبد).

In [ ]:
from sham_data_sources import Ledger, collect

MAX_AUDIO_SAMPLES = 3_000

# جمع بلا تكرار: مصادر موثّقة متعددة بالترتيب (عند نفاد أحدها أو تعطّله يُنتقل للتالي
# تلقائياً)، وموضع استئناف حقيقي لكل مصدر، وبصمة لكل عيّنة سبق جمعها — تُدمج من كل
# المدخلات، فلا يعيد أي دفتر جمع ما جمعه هو أو دفتر آخر من قبل (sham_data_sources.py).
audio_ledger = Ledger.load("audio")
audio_manifest, collect_stats = collect("audio", "/kaggle/working/corpus/audio", MAX_AUDIO_SAMPLES, audio_ledger)
NEW_DATA = collect_stats.written > 0
if not NEW_DATA:
    print("⚠ لا توجد بيانات جديدة غير مكررة — لن يُعاد التدريب على بيانات سبق استخدامها. "
          "إن ظهر أن كل المصادر استُنفدت فالمسار أنهى مهمته، أو أضف مصدراً جديداً إلى SOURCES.")
print(f"جاهز: {audio_manifest}")


### 5) قياس سرعة CPU الحقيقية ثم التدريب ضمن ميزانية زمنية محددة

نفس مبدأ مساري النص: نقيس فعلياً بدل التخمين، ثم نحسب عدد الحقب (epochs) الواقعي ضمن الوقت المتاح.

In [ ]:
if NEW_DATA:
    import time
    import torch
    import soundfile as sf
    from mel_spectrogram import waveform_to_mel_spectrogram
    from train_audio_tokenizer import train_vqvae

    audio_root = Path(audio_manifest).parent
    mels_list = []
    with open(audio_manifest, encoding="utf-8") as f:
        for line in f:
            record = _json.loads(line)
            waveform, sample_rate = sf.read(str(audio_root / record["audio"]), dtype="float32", always_2d=False)
            if waveform.ndim > 1:
                waveform = waveform.mean(axis=1)
            mel = waveform_to_mel_spectrogram(
                torch.from_numpy(waveform), sample_rate, audio_tokenizer_cfg.n_mels, audio_tokenizer_cfg.segment_frames
            )
            mels_list.append(mel)
    real_mels = torch.stack(mels_list, dim=0)
    print(f"عدد المقاطع الصوتية الحقيقية المحمَّلة: {real_mels.shape[0]:,}")

    BATCH_SIZE = 16
    MAX_TRAINING_MINUTES = 45  # هامش أمان آمن ضمن جلسات Kaggle CPU الطويلة، قابل للتعديل

    t0 = time.time()
    _ = train_vqvae(audio_tokenizer, real_mels[: min(BATCH_SIZE, real_mels.shape[0])], num_epochs=1, batch_size=BATCH_SIZE, log_every=999)
    seconds_per_epoch_calib = (time.time() - t0) * (real_mels.shape[0] / min(BATCH_SIZE, real_mels.shape[0]))
    seconds_per_epoch = max(seconds_per_epoch_calib, 0.01)
    num_epochs = max(3, min(60, int((MAX_TRAINING_MINUTES * 60 * 0.85) / seconds_per_epoch)))
    print(f"سرعة حقيقية مقاسة: ~{seconds_per_epoch:.2f} ثانية/حقبة على كامل البيانات -> {num_epochs} حقبة ضمن {MAX_TRAINING_MINUTES} دقيقة.")

    stats = train_vqvae(audio_tokenizer, real_mels, num_epochs=num_epochs, batch_size=BATCH_SIZE, log_every=5)
    print(f"آخر خسارة إعادة بناء+VQ حقيقية: {stats.epoch_losses[-1]:.4f} | استخدام القاموس: {stats.final_codebook_usage}/{stats.codebook_size}")

else:
    stats = None
    print('تخطّي التدريب: لا بيانات جديدة.')


### 6) حفظ ونشر أداة ترميز الصوت (كمجموعة بيانات Kaggle خاصة بهذا المسار)

In [ ]:
from train_audio_tokenizer import save_tokenizer_checkpoint

trained_epochs = len(stats.epoch_losses) if NEW_DATA else 0
final_step = start_step + trained_epochs
final_samples_consumed = samples_consumed + (collect_stats.written if NEW_DATA else 0)

ckpt_dir = Path("/kaggle/working/checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)
save_tokenizer_checkpoint(ckpt_dir / "audio_tokenizer.pt", audio_tokenizer, step=final_step)
(ckpt_dir / "audio_tokenizer_progress.json").write_text(
    _json.dumps({"samples_consumed": final_samples_consumed, "step": final_step, "last_run_sources": collect_stats.per_source})
)
audio_ledger.save(ckpt_dir)  # مواضع الاستئناف + بصمات كل ما جُمع — يرافق نقطة الحفظ دائماً
print(f"تم الحفظ عند الخطوة {final_step:,} (إجمالي العيّنات: {final_samples_consumed:,}) | {collect_stats.summary()}")


In [ ]:
import json as _json
import shutil as _shutil

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=False)

KAGGLE_USERNAME = UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")
DATASET_SLUG = f"{KAGGLE_USERNAME}/sham-audio-tokenizer-checkpoint"

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    _shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True)
_shutil.copy2("/kaggle/working/checkpoints/audio_tokenizer.pt", upload_dir / "audio_tokenizer.pt")
_shutil.copy2("/kaggle/working/checkpoints/audio_tokenizer_progress.json", upload_dir / "audio_tokenizer_progress.json")
_shutil.copy2("/kaggle/working/checkpoints/audio_ledger.json", upload_dir / "audio_ledger.json")

metadata = {"title": "sham-audio-tokenizer-checkpoint", "id": DATASET_SLUG, "licenses": [{"name": "unknown"}]}
(upload_dir / "dataset-metadata.json").write_text(_json.dumps(metadata))

# نفس الفحص الحتمي المعتمد في كل الدفاتر الأخرى: نسأل Kaggle مباشرة هل
# مجموعة البيانات هذه موجودة أصلاً بدل تخمين ذلك من نص رسالة خطأ (غير
# موثّق وقد يتغيّر) — هذا الأسلوب هو ما كشف خطأ "-r skip" الصامت سابقاً.
_list_result = subprocess.run(["kaggle", "datasets", "list", "-m", "--csv"], capture_output=True, text=True)
_dataset_exists = DATASET_SLUG in (_list_result.stdout or "")

# "-r zip" وليس "skip": القيمة الافتراضية الموثّقة لـ Kaggle CLI للمجلدات
# الفرعية هي تجاهلها بصمت، لا رفعها — درس مستفاد من خطأ حقيقي سابق أثّر
# على كل دفاتر التدريب النصية قبل اكتشافه.
if _dataset_exists:
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", f"auto-update at step {final_step:,}", "-r", "zip"],
        capture_output=True, text=True,
    )
else:
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(upload_dir), "-r", "zip"],
        capture_output=True, text=True,
    )
_combined = (result.stdout or "") + (result.stderr or "")

if result.returncode == 0 and "error" not in _combined.lower():
    verb = "تم النشر إلى" if _dataset_exists else "تم إنشاء"
    print(f"{verb} {DATASET_SLUG} — التشغيل المجدول القادم سيلتقطها تلقائياً ويكمل من حيث توقفنا.")
elif "incompatible" in _combined.lower():
    print(
        "تحذير: هذه المجموعة أُنشئت سابقاً بصيغة رفع غير متوافقة. لا يوجد إصلاح على مستوى الكود لها تحديداً — "
        "غيّر الاسم أعلاه لاسم لم يُستخدم من قبل (مثلاً أضف -v2) وأعد التشغيل. نقطة الحفظ نفسها آمنة "
        "في Output هذه الجلسة بغض النظر."
    )
    print(_combined)
else:
    print("تحذير: فشل النشر — نقطة الحفظ لا تزال آمنة في Output هذه الجلسة. تأكد من صحة KAGGLE_USERNAME/KAGGLE_KEY.")
    print(_combined)
